# Twin Check: predicted vs measured on real quantum hardware

This notebook downloads two public, permanent datasets from Zenodo and compares **predictions that were saved before each run**
with **what real 156-qubit superconducting processors measured**.

- Experiments dataset: https://doi.org/10.5281/zenodo.24517586
- Warp-drive analogue paper and data: https://doi.org/10.5281/zenodo.24517295

The predictions were made with LAS-Q1 Studio (Lawrence AI Solutions). **Predictions are published as data; the prediction
engine is not part of this release.** Nothing here recomputes them: the notebook only loads published numbers, so anyone can
check the claims, and anyone can test their own noise model against the same real data.

Independent project; not affiliated with or endorsed by IBM. Code: MIT licence.

In [ ]:
# Runs in Google Colab or any Python 3.9+ with: pip install pandas matplotlib
import io, json, zipfile, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

def fetch_zip(record_id, filename):
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
    with urllib.request.urlopen(url) as r:
        return zipfile.ZipFile(io.BytesIO(r.read()))

DS = fetch_zip(24517587, "pre-registered-quantum-experiments-2026-11.zip")   # experiments dataset, v2.0
WARP = fetch_zip(24517296, "warp-analogue-data.zip")                          # warp-drive analogue data
ROOT = "pre-registered-quantum-experiments-2026-11/data/"
load = lambda z, path: json.loads(z.read(path))
print(len(DS.namelist()), "files in the experiments dataset;", len(WARP.namelist()), "in the warp data")

## 1. Collect every prediction next to its measurement
Each row: experiment, the quantity measured, the prediction (saved in advance), the real chip's value, and the ideal
(perfect, noise-free) value where one exists. All values are probabilities or fractions between 0 and 1.

In [ ]:
rows = []
def add(part, experiment, quantity, predicted, real, ideal=None, chip=""):
    rows.append(dict(part=part, experiment=experiment, quantity=quantity, chip=chip,
                     predicted=float(predicted), real=float(real), ideal=None if ideal is None else float(ideal)))

# Part A (17-18 Nov 2026): the dataset's own accuracy table (success probability of the target answer per circuit)
for r in load(DS, ROOT + "lasq1_accuracy.json")["rows"]:
    add("A", "Part A circuits", r["circuit"], r["lasq1"], r["real"])

# Part B (23-24 Nov 2026), all pre-registered
d = ROOT + "15-zeno-and-teleportation-chain/"
P, R = load(DS, d + "prereg_batch2.json")["studio_prediction"], load(DS, d + "results_batch2.json")
for n in [1, 2, 3, 5, 10, 20]:
    add("B", "Quantum Zeno", f"stayed in 0 after {n} looks", P[f"zeno_{n}"], R["scores"][f"zeno_{n}"], chip="ibm_marrakesh")
for h, real in R["teleport_avg"].items():
    pred = sum(P[f"tele_{h}_{s}"] for s in ["0", "1", "+", "-"]) / 4
    add("B", "Teleportation chain", f"success after {h} hop(s)", pred, real, chip="ibm_marrakesh")

d = ROOT + "16-qft-signal-qaoa-route-noise-assisted-transport/"
P, R = load(DS, d + "prereg_batch3.json")["studio_prediction"], load(DS, d + "results_batch3.json")
add("B", "QFT signal decoder", "share in peaks, period 4", P["signal_r4"], R["scores"]["signal_r4"], 1.0, "ibm_marrakesh")
add("B", "QFT signal decoder", "share in peaks, period 8", P["signal_r8"], R["scores"]["signal_r8"], 1.0, "ibm_marrakesh")
add("B", "QAOA route", "best route found", P["mars_qaoa"], R["scores"]["mars_qaoa"], chip="ibm_marrakesh")
for g, real in R["life"].items():
    keys = [k for k in P if k.startswith(f"life_g{g}_")]
    add("B", "Noise-assisted transport", f"far-end population, dephasing {g}", sum(P[k] for k in keys) / len(keys), real, chip="ibm_marrakesh")

d = ROOT + "17-two-qubit-anticorrelation-wishbone/"
P, R = load(DS, d + "prereg_wishbone.json")["studio_prediction"], load(DS, d + "results_wishbone.json")["summary"]
add("B", "Two-qubit anticorrelation", "exactly one 1", P["wishbone"]["one_winner"], R["wishbone"]["one_winner"], 1.0, "ibm_marrakesh")

d = ROOT + "18-twelve-qubit-ghz-christmas-lights/"
P, R = load(DS, d + "prereg_lights.json")["studio_prediction"], load(DS, d + "results_lights.json")
add("B", "12-qubit GHZ", "all 12 in sync", P["lights"]["in_sync"], R["summary"]["lights"]["in_sync"], 1.0, "ibm_marrakesh")
add("B", "12-qubit GHZ", "fidelity lower bound", P["fidelity_lb"], R["fidelity_lb"], 1.0, "ibm_marrakesh")

# Warp-drive analogue (separate record): light escaping the bubble after the last time step, both chips
P, R = load(WARP, "prereg_warp_studio.json")["predictions"], load(WARP, "results_warp.json")["chips"]
for chip in R:
    for cfg, steps in R[chip]["results"].items():
        last = steps[max(steps, key=int)]
        add("Warp", "Warp-drive analogue", f"{cfg}: light outside", P[chip]["P_out"][cfg], last["P_out"], last.get("ideal_P_out"), chip)

df = pd.DataFrame(rows)
df["miss_points"] = 100 * (df["predicted"] - df["real"])
pd.set_option("display.max_rows", 100)
df.round(3)

## 2. How close were the predictions?
"Points" means percentage points: predicting 75% when the chip measured 77% is a miss of 2 points.

In [ ]:
summary = df.groupby("part")["miss_points"].agg(experiments="count", mean_abs_miss=lambda s: s.abs().mean(), mean_signed=lambda s: s.mean())
print(summary.round(2))
print(f"\nAll {len(df)} comparisons: mean absolute miss {df.miss_points.abs().mean():.1f} points")
print("\nLargest misses (where the model is weakest):")
print(df.reindex(df.miss_points.abs().sort_values(ascending=False).index).head(5)[["experiment", "quantity", "predicted", "real", "miss_points"]].round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for part, g in df.groupby("part"):
    ax.scatter(g["real"], g["predicted"], s=40, label=f"Part {part}" if part != "Warp" else "Warp analogue", alpha=0.85)
ax.plot([0, 1], [0, 1], color="grey", lw=1, ls="--", label="perfect prediction")
ax.set_xlabel("measured on real hardware"); ax.set_ylabel("predicted before the run")
ax.set_title("Predicted vs measured"); ax.legend(); ax.set_aspect("equal"); plt.show()

## 3. Verify the pre-registrations yourself
Each pre-registration stores SHA-256 hashes of the compiled circuits it fixed in advance, and each results file stores the hash
of its pre-registration. This cell recomputes them.

In [ ]:
import hashlib
sha = lambda b: hashlib.sha256(b).hexdigest()
for folder, pre, res, files in [
    ("17-two-qubit-anticorrelation-wishbone", "prereg_wishbone.json", "results_wishbone.json", ["wishbone_isa.json"]),
    ("18-twelve-qubit-ghz-christmas-lights", "prereg_lights.json", "results_lights.json", ["lights_isa.json"]),
    ("16-qft-signal-qaoa-route-noise-assisted-transport", "prereg_batch3.json", "results_batch3.json", ["batch3_isa.json", "batch3_spec.json"]),
]:
    base = ROOT + folder + "/"; p = load(DS, base + pre)
    ok_files = all(sha(DS.read(base + f)) == p["sha256"][f] for f in files)
    ok_link = load(DS, base + res)["prereg_sha256"] == sha(DS.read(base + pre))
    print(f"{folder}: circuits match pre-registration: {ok_files}; results point to this pre-registration: {ok_link}; registered {p['registered_utc']}")

## 4. Optional: a standard open-source baseline (Qiskit Aer)
For comparison, simulate the Bell pair (H then CNOT) with a perfect simulator. Real chips put a little probability on 01 and 10;
a perfect simulator never does. Requires `pip install qiskit qiskit-aer`.

In [ ]:
try:
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
    qc = QuantumCircuit(2, 2); qc.h(0); qc.cx(0, 1); qc.measure([0, 1], [0, 1])
    counts = AerSimulator().run(qc, shots=4096, seed_simulator=1).result().get_counts()
    bell = load(DS, ROOT + "01-bell-and-teleport/lasq1_prediction.json")["bell"]["rows"]
    print("outcome  perfect-simulator  predicted  real chip")
    for r in bell:
        print(f"  {r['k']}        {counts.get(r['k'], 0) / 4096:6.3f}          {r['pred']:6.3f}    {r['real']:6.3f}")
except ImportError:
    print("Install qiskit and qiskit-aer to run this optional baseline.")

## 5. Test your own noise model on the same circuits
The dataset includes the exact chip-compiled circuits that ran on the real processor. This section runs them through a noise
model and scores it the same way as the predictions above. The default model is Qiskit Aer's standard model built from
`FakeMarrakesh` (a stored calibration snapshot shipped with Qiskit). **Replace `my_model` with your own** and re-run.
Requires `pip install qiskit qiskit-aer qiskit-ibm-runtime`; takes a minute or two.

In [ ]:
# In Google Colab, first run:  !pip install qiskit qiskit-aer qiskit-ibm-runtime
import numpy as np
from qiskit import QuantumCircuit, qasm2, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
backend = FakeMarrakesh()                      # a stored snapshot of the chip's calibration, shipped with Qiskit
SHOTS = 20000

def on_chip(qasm_chip, physical, n_clbits):
    "Place a published chip-compiled circuit onto its physical qubits."
    small = qasm2.loads(qasm_chip, custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)
    big = QuantumCircuit(backend.num_qubits, n_clbits)
    big.compose(small, qubits=physical, clbits=list(range(n_clbits)), inplace=True)
    return big

In [ ]:
# Replace this function with your own noise model. It receives a circuit placed on the chip's physical qubits and must
# return measurement counts {bitstring: count}. The default is Qiskit Aer's standard model built from the fake backend.
_sim = AerSimulator.from_backend(backend)
def my_model(circuit, shots=SHOTS):
    return _sim.run(circuit, shots=shots, seed_simulator=7).result().get_counts()

In [ ]:
scores = []
def score(experiment, quantity, model_value):
    row = df[(df.experiment == experiment) & (df.quantity == quantity)].iloc[0]
    scores.append(dict(experiment=experiment, quantity=quantity, real=row.real, lasq1=row.predicted, your_model=model_value))

d = ROOT + "17-two-qubit-anticorrelation-wishbone/"
r = [c for c in load(DS, d + "wishbone_isa.json")["circuits"] if c["name"] == "wishbone"][0]
c = my_model(on_chip(r["qasm_chip"], r["physical"], 2)); n = sum(c.values())
score("Two-qubit anticorrelation", "exactly one 1", (c.get("01", 0) + c.get("10", 0)) / n)

d = ROOT + "18-twelve-qubit-ghz-christmas-lights/"
I = {c["name"]: c for c in load(DS, d + "lights_isa.json")["circuits"]}
N = load(DS, d + "lights_isa.json")["n"]
cl = my_model(on_chip(I["lights"]["qasm_chip"], I["lights"]["physical"], N)); n = sum(cl.values())
sync = (cl.get("0" * N, 0) + cl.get("1" * N, 0)) / n
ct = my_model(on_chip(I["twinkle"]["qasm_chip"], I["twinkle"]["physical"], N)); n = sum(ct.values())
parity = sum(v * (1 if k.count("1") % 2 == 0 else -1) for k, v in ct.items()) / n
score("12-qubit GHZ", "all 12 in sync", sync)
score("12-qubit GHZ", "fidelity lower bound", (sync + parity) / 2)

d = ROOT + "15-zeno-and-teleportation-chain/"
path = load(DS, d + "batch2_isa.json")["path"]
for sp in load(DS, d + "batch2_spec.json"):
    if not sp["name"].startswith("zeno_") or sp["name"] == "zeno_unwatched": continue
    qc = QuantumCircuit(sp["n"], sp["nc"])
    for op in sp["ops"]:
        if op["name"] == "ry": qc.ry(op["params"][0], op["qubits"][0])
        elif op["name"] == "measure": qc.measure(op["qubits"][0], op["clbits"][0])
    t = transpile(qc, backend, initial_layout=[path[0]], optimization_level=1, seed_transpiler=7)
    c = my_model(t); n = sum(c.values())
    score("Quantum Zeno", f"stayed in 0 after {sp['nc']} looks", c.get("0" * sp["nc"], 0) / n)

S = pd.DataFrame(scores)
S["miss_lasq1"] = 100 * (S.lasq1 - S.real); S["miss_your_model"] = 100 * (S.your_model - S.real)
print(S.round(3).to_string(index=False))
print(f"\nMean absolute miss, LAS-Q1 (saved before the run): {S.miss_lasq1.abs().mean():.1f} points")
print(f"Mean absolute miss, your model:                    {S.miss_your_model.abs().mean():.1f} points")
print("\nNote: the fake backend's calibration is a stored snapshot, not the calibration on the day of each run,"
      " so a fair comparison would also use that day's data. Try improving the model and see if you can beat both.")

## Use this data for your own model
Every folder in the dataset includes the chip-compiled circuits (OpenQASM 2) and raw counts. Predict them with your own noise
model, add your predictions as a new column above, and compare. If you publish a comparison, please cite both DOIs.